# Warehouse Schema & Data Structure Exploration

This notebook provides detailed exploration of each table in the warehouse:
- Column names and types
- Row counts and grain verification
- Missing value patterns (NULL and ZERO)
- Sample data and value distributions
- Panel structure and history windows

In [3]:
%pip -q install duckdb huggingface_hub pandas

Note: you may need to restart the kernel to use updated packages.


In [4]:
import os
import getpass
import pandas as pd
import duckdb
from datetime import datetime

# Token setup (same as notebook 03)
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token: ')

# Connect to warehouse
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':           f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':           f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':            f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':     f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':        f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

print('✓ Connected to warehouse')

✓ Connected to warehouse


## 1. Basic Counts & Date Ranges

In [5]:
# Get row counts for all tables
counts = {}
for name, src in TABLES.items():
    query = f"SELECT COUNT(*) as cnt FROM {src}"
    result = con.sql(query).df()
    counts[name] = result.iloc[0]['cnt']
    
    # Also check date range if report_date exists
    try:
        date_query = f"SELECT MIN(report_date) as min_date, MAX(report_date) as max_date FROM {src}"
        date_result = con.sql(date_query).df()
        min_date = date_result.iloc[0]['min_date']
        max_date = date_result.iloc[0]['max_date']
        print(f"{name:25} {counts[name]:>15,} rows   |  {min_date} to {max_date}")
    except:
        print(f"{name:25} {counts[name]:>15,} rows")

dim_clients                           104 rows
dim_content                       519,606 rows
fact_daily                     78,835,655 rows   |  2025-01-27 00:00:00 to 2026-06-30 00:00:00
fact_daily_sample              11,694,072 rows   |  2026-06-01 00:00:00 to 2026-06-30 00:00:00
fact_query_90d                  2,414,248 rows


## 2. Detailed Schema (All Columns & Types)

In [14]:
# Get schema for each table
schemas = {}
for name, src in TABLES.items():
    # Build schema result from an empty dataframe
    rel_df = con.sql(f"SELECT * FROM {src} LIMIT 0").df()
    def _sql_type_from_pd(dtype):
        s = str(dtype).lower()
        if 'int' in s: return 'BIGINT'
        if 'float' in s: return 'DOUBLE'
        if 'datetime' in s or 'timestamp' in s: return 'TIMESTAMP'
        if 'bool' in s: return 'BOOLEAN'
        return 'VARCHAR'
    schema_result = pd.DataFrame([{'column_name': c, 'type': _sql_type_from_pd(t)} for c, t in rel_df.dtypes.items()])
    schemas[name] = schema_result
    
    print(f"\n{'='*80}")
    print(f"{name.upper()} - {counts[name]:,} rows")
    print(f"{'='*80}")
    pd.set_option('display.max_colwidth', 50)
    print(schema_result.to_string(index=False))


DIM_CLIENTS - 104 rows
        column_name      type
     client_hash_id   VARCHAR
          is_active   BOOLEAN
     has_gsc_access   BOOLEAN
     has_ga4_access   BOOLEAN
     access_profile   VARCHAR
client_created_date TIMESTAMP
client_updated_date TIMESTAMP
     gsc_data_start TIMESTAMP
     ga4_data_start TIMESTAMP

DIM_CONTENT - 519,606 rows
               column_name      type
            client_hash_id   VARCHAR
           content_hash_id   VARCHAR
           keyword_hash_id   VARCHAR
               url_hash_id   VARCHAR
        keyword_char_count    BIGINT
       keyword_token_count    BIGINT
            url_char_count    BIGINT
      content_created_date TIMESTAMP
      content_updated_date TIMESTAMP
              content_type   VARCHAR
             search_volume    BIGINT
               competition    DOUBLE
         competition_level   VARCHAR
                       cpc    DOUBLE
               main_intent   VARCHAR
                 backlinks    BIGINT
            categor

## 3. Detailed Column Exploration for fact_daily

In [16]:
# Deep dive into fact_daily schema - this is the largest table
print("\nFACT_DAILY - Detailed Column Analysis")
print("="*80)

# Build fact_schema from a zero-row sample
rel_df = con.sql(f"SELECT * FROM {TABLES['fact_daily']} LIMIT 0").df()
def _sql_type_from_pd(dtype):
    s = str(dtype).lower()
    if 'int' in s: return 'BIGINT'
    if 'float' in s: return 'DOUBLE'
    if 'datetime' in s or 'timestamp' in s: return 'TIMESTAMP'
    if 'bool' in s: return 'BOOLEAN'
    return 'VARCHAR'
fact_schema = pd.DataFrame([{'column_name': c, 'type': _sql_type_from_pd(t)} for c, t in rel_df.dtypes.items()])

# Separate by type
numeric_cols = fact_schema[fact_schema['type'].str.contains('INTEGER|DOUBLE|FLOAT|BIGINT|DECIMAL', case=False)]['column_name'].tolist()
date_cols = fact_schema[fact_schema['type'].str.contains('DATE|TIMESTAMP', case=False)]['column_name'].tolist()
string_cols = fact_schema[fact_schema['type'].str.contains('VARCHAR|TEXT|STRING', case=False)]['column_name'].tolist()
bool_cols = fact_schema[fact_schema['type'].str.contains('BOOL', case=False)]['column_name'].tolist()

print(f"\nNumeric columns ({len(numeric_cols)}):")
print(f"  {', '.join(numeric_cols)}")

print(f"\nDate/Timestamp columns ({len(date_cols)}):")
print(f"  {', '.join(date_cols)}")

print(f"\nString/ID columns ({len(string_cols)}):")
print(f"  {', '.join(string_cols)}")

print(f"\nBoolean columns ({len(bool_cols)}):")
print(f"  {', '.join(bool_cols)}")


FACT_DAILY - Detailed Column Analysis

Numeric columns (23):
  gsc_impressions, gsc_clicks, gsc_sum_position, gsc_avg_position, ga4_pageviews, ga4_sessions, ga4_users, ga4_engaged_sessions, ga4_total_engagement_sec, sessions_organic, sessions_direct, sessions_referral, sessions_social, sessions_paid, sessions_ai, ai_chatgpt, ai_perplexity, ai_gemini, ai_copilot, ai_claude, ai_meta, ai_other, scroll_events

Date/Timestamp columns (1):
  report_date

String/ID columns (3):
  client_hash_id, content_hash_id, month

Boolean columns (4):
  client_has_gsc, client_has_ga4, gsc_data_available, ga4_data_available


## 4. NULL Values Analysis

In [17]:
# Analyze NULL patterns in fact_daily
print("\nFACT_DAILY - NULL Value Patterns")
print("="*80)

# Build schema from zero-row sample (avoid DESCRIBE)
rel_df = con.sql(f"SELECT * FROM {TABLES['fact_daily']} LIMIT 0").df()
def _sql_type_from_pd(dtype):
    s = str(dtype).lower()
    if 'int' in s: return 'BIGINT'
    if 'float' in s: return 'DOUBLE'
    if 'datetime' in s or 'timestamp' in s: return 'TIMESTAMP'
    if 'bool' in s: return 'BOOLEAN'
    return 'VARCHAR'
fact_schema = pd.DataFrame([{'column_name': c, 'type': _sql_type_from_pd(t)} for c, t in rel_df.dtypes.items()])
all_cols = fact_schema['column_name'].tolist()

# Build dynamic NULL checks
null_checks = []
for col in all_cols:
    query = f"""
    SELECT 
        '{col}' as column_name,
        COUNT(*) as total_rows,
        COUNT(CASE WHEN {col} IS NULL THEN 1 END) as null_count,
        ROUND(100.0 * COUNT(CASE WHEN {col} IS NULL THEN 1 END) / COUNT(*), 2) as null_pct
    FROM {TABLES['fact_daily']}
    """
    try:
        result = con.sql(query).df().iloc[0]
        null_checks.append(result)
    except:
        pass

null_df = pd.DataFrame(null_checks)
# Show only columns with NULLs
null_with_nulls = null_df[null_df['null_count'] > 0].sort_values('null_pct', ascending=False)

if len(null_with_nulls) > 0:
    print("\nColumns with NULL values:")
    print(null_with_nulls[['column_name', 'null_count', 'null_pct']].to_string(index=False))
else:
    print("\n✓ No NULL values in any column")


FACT_DAILY - NULL Value Patterns

Columns with NULL values:
             column_name  null_count  null_pct
        gsc_avg_position    49865654     63.25
               ga4_users    29635327     37.59
      ga4_data_available    29635327     37.59
           ga4_pageviews    29635327     37.59
            ga4_sessions    29635327     37.59
         sessions_direct    29635327     37.59
        sessions_organic    29635327     37.59
ga4_total_engagement_sec    29635327     37.59
    ga4_engaged_sessions    29635327     37.59
              ai_chatgpt    29635327     37.59
           ai_perplexity    29635327     37.59
               ai_gemini    29635327     37.59
              ai_copilot    29635327     37.59
       sessions_referral    29635327     37.59
         sessions_social    29635327     37.59
           sessions_paid    29635327     37.59
             sessions_ai    29635327     37.59
               ai_claude    29635327     37.59
                 ai_meta    29635327     37.59

## 5. ZERO Values in Numeric Columns

In [18]:
# Check for zero counts in numeric columns
print("\nFACT_DAILY - ZERO Value Patterns (Numeric Columns)")
print("="*80)

# Define key numeric metrics that should be checked
key_metrics = [
    'gsc_impressions', 'gsc_clicks', 'gsc_avg_position',
    'ga4_sessions', 'ga4_users', 'ga4_engagement_sessions',
    'ga4_engaged_sessions', 'ga4_engagement_rate'
]

zero_checks = []
for col in key_metrics:
    query = f"""
    SELECT 
        '{col}' as column_name,
        COUNT(CASE WHEN {col} = 0 THEN 1 END) as zero_count,
        ROUND(100.0 * COUNT(CASE WHEN {col} = 0 THEN 1 END) / COUNT(*), 2) as zero_pct,
        MIN({col}) as min_val,
        MAX({col}) as max_val
    FROM {TABLES['fact_daily']}
    """
    try:
        result = con.sql(query).df().iloc[0]
        zero_checks.append(result)
    except:
        pass

zero_df = pd.DataFrame(zero_checks)
print(zero_df.to_string(index=False))


FACT_DAILY - ZERO Value Patterns (Numeric Columns)
         column_name  zero_count  zero_pct  min_val  max_val
     gsc_impressions    49767598     63.13      0.0 245826.0
          gsc_clicks    75625042     95.93      0.0   9558.0
    gsc_avg_position     1393857      1.77      0.0    907.0
        ga4_sessions    46421764     58.88      0.0  19425.0
           ga4_users    46421764     58.88      0.0  33755.0
ga4_engaged_sessions    48995553     62.15      0.0    234.0


## 6. Grain Verification (Check for Duplicates)

In [20]:
# Verify grain for each table
print("\nGRAIN VERIFICATION - Checking for Duplicate Rows")
print("="*80)

grain_checks = {
    'dim_clients': ['client_hash_id'],
    'dim_content': ['content_hash_id'],
    'fact_daily': ['report_date', 'client_hash_id', 'content_hash_id'],
    'fact_daily_sample': ['report_date', 'client_hash_id', 'content_hash_id'],
    'fact_query_90d': ['client_hash_id', 'content_hash_id', 'query_hash_id'],
}

for table_name, grain_cols in grain_checks.items():
    grain_str = ', '.join(grain_cols)
    query = f"""
    WITH grain_counts AS (
        SELECT {grain_str}, COUNT(*) as cnt
        FROM {TABLES[table_name]}
        GROUP BY {grain_str}
    )
    SELECT 
        COUNT(*) as total_grains,
        COUNT(CASE WHEN cnt > 1 THEN 1 END) as duplicate_grains,
        MAX(cnt) as max_rows_per_grain,
        SUM(CASE WHEN cnt > 1 THEN cnt - 1 ELSE 0 END) as extra_rows
    FROM grain_counts
    """
    
    result = con.sql(query).df().iloc[0]
    print(f"\n{table_name}:")
    print(f"  Grain: ({grain_str})")
    print(f"  Total distinct grains: {result['total_grains']:,}")
    print(f"  Duplicate grains: {result['duplicate_grains']:,}")
    print(f"  Extra rows from duplication: {result['extra_rows']:,}")
    
    if result['duplicate_grains'] > 0:
        print(f"  ⚠️  WARNING: This table has duplicate rows!")
        # Show sample
        sample_query = f"""
        SELECT {grain_str}, COUNT(*) as cnt
        FROM {TABLES[table_name]}
        GROUP BY {grain_str}
        HAVING COUNT(*) > 1
        LIMIT 5
        """
        samples = con.sql(sample_query).df()
        print(f"\n  Sample duplicated grains:")
        print(samples.to_string(index=False))


GRAIN VERIFICATION - Checking for Duplicate Rows

dim_clients:
  Grain: (client_hash_id)
  Total distinct grains: 104.0
  Duplicate grains: 0.0
  Extra rows from duplication: 0.0

dim_content:
  Grain: (content_hash_id)
  Total distinct grains: 519,606.0
  Duplicate grains: 0.0
  Extra rows from duplication: 0.0

fact_daily:
  Grain: (report_date, client_hash_id, content_hash_id)
  Total distinct grains: 78,829,265.0
  Duplicate grains: 6,390.0
  Extra rows from duplication: 6,390.0
  ⚠️  WARNING: This table has duplicate rows!

  Sample duplicated grains:
report_date          client_hash_id          content_hash_id  cnt
 2026-06-30 client_def0955f7a377868 content_fa5e8c5d22b24cf0    2
 2026-06-30 client_e00b29e582949543 content_ac3831ae8497a17e    2
 2026-06-15 client_810019792c9b8efc content_a44a3f299cba632b    2
 2026-06-15 client_a22068e339bf95f5 content_69d9c3ab3c89bbb2    2
 2026-06-19 client_b77d0d5f08f05e64 content_3d9fc8e76439c31e    2

fact_daily_sample:
  Grain: (report_dat

## 7. DIM_CLIENTS - Panel Structure

In [21]:
# Explore the client panel structure
print("\nDIM_CLIENTS - Panel Structure & History")
print("="*80)

clients = con.sql(f"""
    SELECT * FROM {TABLES['dim_clients']}
    ORDER BY gsc_data_start ASC
""").df()

print(f"\nTotal clients: {len(clients)}")
print(f"\nAccess profiles:")
print(clients['access_profile'].value_counts())

print(f"\nGSC data start dates:")
print(f"  Earliest: {clients['gsc_data_start'].min()}")
print(f"  Latest: {clients['gsc_data_start'].max()}")
print(f"  Unique dates: {clients['gsc_data_start'].nunique()}")

print(f"\nGA4 data start dates:")
ga4_dates = clients[clients['ga4_data_start'].notna()]['ga4_data_start']
print(f"  Clients with GA4: {len(ga4_dates)}")
if len(ga4_dates) > 0:
    print(f"  Earliest: {ga4_dates.min()}")
    print(f"  Latest: {ga4_dates.max()}")

print(f"\nGSC-only clients: {(clients['access_profile'] == 'gsc_only').sum()}")

print("\nFirst 10 clients (ordered by GSC start date):")
print(clients[['client_hash_id', 'access_profile', 'gsc_data_start', 'ga4_data_start']].head(10).to_string(index=False))


DIM_CLIENTS - Panel Structure & History

Total clients: 104

Access profiles:
access_profile
gsc_and_ga4                             53
no_search_or_analytics_access           26
gsc_only                                14
source_only_missing_client_dimension    10
ga4_only                                 1
Name: count, dtype: int64

GSC data start dates:
  Earliest: 2025-01-27 00:00:00
  Latest: 2026-06-02 00:00:00
  Unique dates: 45

GA4 data start dates:
  Clients with GA4: 51
  Earliest: 2025-10-29 00:00:00
  Latest: 2026-06-01 00:00:00

GSC-only clients: 14

First 10 clients (ordered by GSC start date):
         client_hash_id access_profile gsc_data_start ga4_data_start
client_9958f0a7ae1df715    gsc_and_ga4     2025-01-27     2025-10-29
client_ff644d8251367cbb    gsc_and_ga4     2025-01-27     2025-10-29
client_73cda7b4e4f265ea    gsc_and_ga4     2025-02-11     2026-03-24
client_fef1a8f436438636    gsc_and_ga4     2025-03-11     2026-03-06
client_62f4a7e64f5e0096       gsc_only 

## 8. DIM_CONTENT - Content Metadata

In [22]:
# Explore content dimension
print("\nDIM_CONTENT - Content Items & Metadata")
print("="*80)

# Get sample
content_sample = con.sql(f"SELECT * FROM {TABLES['dim_content']} LIMIT 5").df()
print(f"\nTotal unique content items: {counts['dim_content']:,}")
print(f"\nFirst 5 content items:")
print(content_sample.to_string(index=False))

# Check for any NULL or zero values in key columns
print(f"\n\nContent metadata summary:")
content_describe = con.sql(f"""
    SELECT 
        COUNT(*) as total,
        COUNT(DISTINCT content_hash_id) as unique_ids,
        COUNT(DISTINCT client_hash_id) as clients_with_content
    FROM {TABLES['dim_content']}
""").df()
print(content_describe.to_string(index=False))


DIM_CONTENT - Content Items & Metadata

Total unique content items: 519,606

First 5 content items:
         client_hash_id          content_hash_id          keyword_hash_id          url_hash_id  keyword_char_count  keyword_token_count  url_char_count content_created_date content_updated_date    content_type  search_volume  competition competition_level  cpc   main_intent  backlinks  category_count keyword_created_date           provider_used             model_used  char_count  word_count last_optimized_date optimization_eligible_date  is_published  is_deleted
client_04660893ae39614a content_004de9653278b5a4 keyword_e754999ab88dd9f2 url_d6091f18cf628794                  22                    4             108           2026-05-30           2026-07-01 keyword article             30         0.91              HIGH 0.98 transactional         16               3           2026-05-12 gemini-generate-content gemini-3-flash-preview       15682        2555                 NaT                   

## 9. FACT_QUERY_90D - Query-Level Data

In [27]:
# Explore query-level fact table
print("\nFACT_QUERY_90D - Query-Level Data")
print("="*80)

q_src = TABLES['fact_query_90d']
q_cols = con.sql(f"SELECT * FROM {q_src} LIMIT 0").df().columns.tolist()

def pick_col(candidates, required=True):
    for c in candidates:
        if c in q_cols:
            return c
    if required:
        raise ValueError(f"None of {candidates} found. Available columns: {q_cols}")
    return None

query_key_col = pick_col(['query_hash_id', 'query_hash'])
impr_col = pick_col(['impressions', 'gsc_impressions', 'impressions_90d', 'impressions_last30', 'impressions_prev30'])
click_col = pick_col(['clicks', 'gsc_clicks', 'clicks_90d', 'clicks_last30', 'clicks_prev30'])
vol_col = pick_col(['query_volume', 'query_volume_90d', 'search_volume', 'query_search_volume'], required=False)

print("\nColumns:")
print(", ".join(q_cols))
print(f"\nUsing query key column: {query_key_col}")
print(f"Using impressions column: {impr_col}")
print(f"Using clicks column: {click_col}")
if vol_col:
    print(f"Using query-volume context column: {vol_col}")
else:
    print("No query-volume context column found (this is okay).")

summary = con.sql(f"""
    WITH pair_counts AS (
        SELECT client_hash_id, content_hash_id, COUNT(*) as queries_per_pair
        FROM {q_src}
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT
        (SELECT COUNT(*) FROM {q_src}) as total_rows,
        (SELECT COUNT(DISTINCT client_hash_id) FROM {q_src}) as distinct_clients,
        (SELECT COUNT(DISTINCT content_hash_id) FROM {q_src}) as distinct_content,
        (SELECT COUNT(DISTINCT client_hash_id || '|' || content_hash_id) FROM {q_src}) as distinct_client_content_pairs,
        (SELECT COUNT(DISTINCT {query_key_col}) FROM {q_src}) as distinct_queries,
        MIN(queries_per_pair) as min_queries_per_pair,
        AVG(queries_per_pair) as avg_queries_per_pair,
        MAX(queries_per_pair) as max_queries_per_pair
    FROM pair_counts
""").df()

print("\nQuery-table structure summary:")
print(summary.to_string(index=False))

sample_extra_col = f", {vol_col} as query_volume" if vol_col else ""
sample = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        {query_key_col} as query_key,
        {impr_col} as impressions,
        {click_col} as clicks
        {sample_extra_col}
    FROM {q_src}
    ORDER BY {impr_col} DESC NULLS LAST
    LIMIT 5
""").df()

print("\nTop 5 query rows by impressions:")
print(sample.to_string(index=False))

agg_volume = f", ANY_VALUE({vol_col}) as any_query_volume" if vol_col else ""
agg = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM({impr_col}) as total_impressions,
        SUM({click_col}) as total_clicks,
        COUNT(DISTINCT {query_key_col}) as n_queries
        {agg_volume}
    FROM {q_src}
    GROUP BY client_hash_id, content_hash_id
    ORDER BY total_impressions DESC
    LIMIT 5
""").df()

print("\nTop 5 client-content pairs after query-level aggregation:")
print(agg.to_string(index=False))

print("\nAggregation reminder:")
print("- SUM() additive metrics (impressions/clicks)")
print("- COUNT(DISTINCT query_key) for query breadth")
print("- ANY_VALUE() for repeated context fields like query_volume")


FACT_QUERY_90D - Query-Level Data

Columns:
client_hash_id, content_hash_id, query_hash_id, query_char_count, query_token_count, window_start, window_end, impressions_90d, clicks_90d, impressions_last30, clicks_last30, impressions_prev30, clicks_prev30, avg_position_90d, avg_position_last30, avg_position_prev30, content_total_impressions_90d, content_visible_query_count, rare_query_count, rare_impressions_share, anonymized_impressions_share

Using query key column: query_hash_id
Using impressions column: impressions_90d
Using clicks column: clicks_90d
No query-volume context column found (this is okay).

Query-table structure summary:
 total_rows  distinct_clients  distinct_content  distinct_client_content_pairs  distinct_queries  min_queries_per_pair  avg_queries_per_pair  max_queries_per_pair
    2414248                52            133852                         133852           1180090                     1             18.036697                  7889

Top 5 query rows by impressio

## 10. Column-by-Column Reference for fact_daily

In [23]:
# Create a comprehensive column reference table
print("\nCOMPREHENSIVE COLUMN REFERENCE - fact_daily")
print("="*80)

# Build fact_cols from zero-row sample (avoid DESCRIBE)
rel_df = con.sql(f"SELECT * FROM {TABLES['fact_daily']} LIMIT 0").df()
def _sql_type_from_pd(dtype):
    s = str(dtype).lower()
    if 'int' in s: return 'BIGINT'
    if 'float' in s: return 'DOUBLE'
    if 'datetime' in s or 'timestamp' in s: return 'TIMESTAMP'
    if 'bool' in s: return 'BOOLEAN'
    return 'VARCHAR'
fact_cols = pd.DataFrame([{'column_name': c, 'type': _sql_type_from_pd(t)} for c, t in rel_df.dtypes.items()])

# Build reference with stats
reference = []
for _, row in fact_cols.iterrows():
    col_name = row['column_name']
    col_type = row['type']
    
    # Get stats
    stats_query = f"""
    SELECT 
        '{col_name}' as column_name,
        COUNT(CASE WHEN {col_name} IS NULL THEN 1 END) as null_count,
        COUNT(DISTINCT {col_name}) as unique_values
    FROM {TABLES['fact_daily']}
    """
    try:
        stats = con.sql(stats_query).df().iloc[0]
        reference.append({
            'column': col_name,
            'type': col_type,
            'nulls': stats['null_count'],
            'unique_values': stats['unique_values']
        })
    except:
        reference.append({
            'column': col_name,
            'type': col_type,
            'nulls': 'N/A',
            'unique_values': 'N/A'
        })

ref_df = pd.DataFrame(reference)
print(ref_df.to_string(index=False))


COMPREHENSIVE COLUMN REFERENCE - fact_daily
                  column      type    nulls  unique_values
             report_date TIMESTAMP        0            520
          client_hash_id   VARCHAR        0             70
         content_hash_id   VARCHAR        0         427292
          client_has_gsc   BOOLEAN        0              2
          client_has_ga4   BOOLEAN        0              2
      gsc_data_available   BOOLEAN    98006              2
      ga4_data_available   BOOLEAN 29635327              2
         gsc_impressions    BIGINT    98006           7917
              gsc_clicks    BIGINT    98006            271
        gsc_sum_position    BIGINT    98014          50194
        gsc_avg_position    DOUBLE 49865654        1364674
           ga4_pageviews    BIGINT 29635327            635
            ga4_sessions    BIGINT 29635327            501
               ga4_users    BIGINT 29635327            498
    ga4_engaged_sessions    BIGINT 29635327             67
ga4_total_e

## 11. Key Gotchas Summary

In [24]:
print("""
KEY GOTCHAS & INSIGHTS FROM THE WAREHOUSE
==========================================

1. **Grain Issues:**
   - fact_daily: One row per (report_date, client_hash_id, content_hash_id)
   - CHECK if there are duplicates per grain (indicates data quality issue)
   - If duplicates exist, aggregate or filter before modeling

2. **Position Column Gotcha:**
   - gsc_avg_position = 0 means "NO DATA", not rank zero
   - Must filter or create a has_position flag

3. **GA4 Data Availability:**
   - Check ga4_data_available column
   - Rows with ga4_data_available=FALSE have zeros, not "no engagement"
   - Filter on ga4_data_available before using GA4 features

4. **Panel Imbalance:**
   - Clients have vastly different history depths
   - Some clients: GSC-only (no GA4)
   - Some clients: GA4 starts much later than GSC
   - Use per-client time windows, not one global window

5. **Query Table Grain:**
   - fact_query_90d: Multiple rows per (client, content) pair
   - One row per QUERY within each client-content pair
   - When aggregating, use ANY_VALUE() for repeated context columns

6. **Column Names:**
   - Warehouse uses: gsc_impressions, gsc_clicks, gsc_avg_position
   - Starter CSV used: impressions, clicks, avg_position
   - Data contract must use warehouse names

7. **Sample vs. Full:**
   - fact_daily_sample = June 2026 only (final month of panel)
   - Use _sample for query testing/mechanics only
   - Use full fact_daily for final modeling
   - Avoid label leakage: don't build labels inside June 2026 data

NEXT STEPS:
1. Review the column lists and NULL/ZERO patterns above
2. Update work/notebooks/w03_data_contract.ipynb with:
   - Exact grain definition
   - Exact column mappings (warehouse names)
   - Filtered datasets (e.g., ga4_data_available=TRUE)
   - Time windows per client
3. Verify no duplicate grains in fact_daily
4. Define how to handle gsc_avg_position=0
""")


KEY GOTCHAS & INSIGHTS FROM THE WAREHOUSE

1. **Grain Issues:**
   - fact_daily: One row per (report_date, client_hash_id, content_hash_id)
   - CHECK if there are duplicates per grain (indicates data quality issue)
   - If duplicates exist, aggregate or filter before modeling

2. **Position Column Gotcha:**
   - gsc_avg_position = 0 means "NO DATA", not rank zero
   - Must filter or create a has_position flag

3. **GA4 Data Availability:**
   - Check ga4_data_available column
   - Rows with ga4_data_available=FALSE have zeros, not "no engagement"
   - Filter on ga4_data_available before using GA4 features

4. **Panel Imbalance:**
   - Clients have vastly different history depths
   - Some clients: GSC-only (no GA4)
   - Some clients: GA4 starts much later than GSC
   - Use per-client time windows, not one global window

5. **Query Table Grain:**
   - fact_query_90d: Multiple rows per (client, content) pair
   - One row per QUERY within each client-content pair
   - When aggregating, 